In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# City configuration
city_name = "Sao Paulo"
latitude = -23.5505
longitude = -46.6333

# Date range
start_date = "2024-01-01"
end_date = "2024-12-31"

# Open-Meteo historical weather API URL
url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={latitude}"
    f"&longitude={longitude}"
    f"&start_date={start_date}"
    f"&end_date={end_date}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,windspeed_10m_max"
    "&timezone=America%2FSao_Paulo"
)

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"API request failed: {response.status_code} - {response.text}")

data = response.json()

# Convert daily weather data into Pandas DataFrame
daily_data = data["daily"]

df = pd.DataFrame({
    "date": daily_data["time"],
    "temperature_max": daily_data["temperature_2m_max"],
    "temperature_min": daily_data["temperature_2m_min"],
    "temperature_mean": daily_data["temperature_2m_mean"],
    "precipitation_sum": daily_data["precipitation_sum"],
    "windspeed_max": daily_data["windspeed_10m_max"]
})

# Add metadata
df["city"] = city_name
df["latitude"] = latitude
df["longitude"] = longitude
df["source"] = "Open-Meteo API"

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Convert Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Create catalog and schema
spark.sql("CREATE CATALOG IF NOT EXISTS portfolio")
spark.sql("CREATE SCHEMA IF NOT EXISTS portfolio.weather_api_pipeline")

# Add ingestion timestamp
spark_df = spark_df.withColumn("ingestion_timestamp", current_timestamp())

# Save raw weather data as Delta table
spark_df.write.mode("overwrite").saveAsTable(
    "portfolio.weather_api_pipeline.raw_weather_sao_paulo"
)

display(spark_df)